# 3. Refit the Senate models

This notebook runs new optimizations using the bundled data. It defaults to **smoke mode**, a short software check. Smoke results are not manuscript results and must not be reported as a reproduction of the full analysis.

Set the environment variable `IDEALPOINT_MODE=full` before executing the notebook to run the full fitting protocol. Full mode covers both exponents and all three coordinate-prior scales, with the prescribed starts and refinements. It can take substantially longer than replaying the saved reference results.

Select the intended mode in the configuration cell below. See the README for environment setup.

Each Run All creates a new timestamped directory below `outputs/`. To repeat the fitting cell, first rerun the configuration cell to create a new output name. The saved manuscript inputs in `reference/` are preserved. Run `1_reproduce_manuscript.ipynb` if you only want the reported tables and figures.


In [ ]:
from pathlib import Path
import json
import os
import sys

# Works when Jupyter starts in the repository root or a directory below it.
working_directory = Path.cwd().resolve()
ROOT = next(
    (candidate for candidate in (working_directory, *working_directory.parents)
     if (candidate / "idealpoint").is_dir()
     and (candidate / "reference").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the replication repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA = ROOT / "data"
REFERENCE = ROOT / "reference"
os.environ.setdefault("MPLBACKEND", "Agg")
cache_directory = ROOT / "outputs" / "_matplotlib"
cache_directory.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(cache_directory))

import numpy as np
import pandas as pd
from IPython.display import Image, JSON, Markdown, display

print("Repository:", ROOT.name)


## 1. Select the run mode

The default is deliberately short. A full run follows the recorded study design and reports the selected fits at all scales. Model starts are selected using the penalized training objective, never the held-out scores.


In [ ]:
MODE = os.environ.get("IDEALPOINT_MODE", "smoke").strip().lower()
# For an intentional interactive full run, change MODE to "full" here.
if MODE not in {"smoke", "full"}:
    raise ValueError("IDEALPOINT_MODE must be 'smoke' or 'full'.")

from datetime import datetime, timezone
run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT = ROOT / "outputs" / f"{MODE}-refit-{run_stamp}"
print("Run mode:", MODE)
print("Output directory:", OUTPUT.relative_to(ROOT))
if MODE == "smoke":
    print("SMOKE TEST ONLY: these estimates do not reproduce the manuscript fitting protocol.")
else:
    print("Full fitting protocol selected. Numerical outputs can vary across software and hardware.")


## 2. Run the fitting procedure

The study uses the exact Euclidean norm for the unsquared model. It does not smooth the norm near coincidence. The resulting optimization problem is nonsmooth, so solver termination is not a certificate of stationarity or global optimality.

Runtime below measures this execution on the current machine. It is separate from the timings stored with the historical reference run. The returned report and output files record the new run.


In [ ]:
from time import perf_counter
from idealpoint.study import run

started = perf_counter()
run_report = run(mode=MODE, output_dir=OUTPUT)
elapsed_seconds = perf_counter() - started
print(f"Elapsed time for this {MODE} run: {elapsed_seconds:.1f} seconds")
display(JSON(run_report, expanded=False))


## 3. Inspect the new scores and diagnostics

Lower log loss and Brier score indicate better prediction of the held-out entries in this split. These scores alone do not validate an ideological interpretation of the coordinates. Equal parameter priors under the two exponents also induce different priors on probabilities.

In smoke mode, this table is a check that the fitting and scoring pipeline runs. Use only a full run when assessing numerical agreement with the manuscript.


In [ ]:
fresh_scores = pd.read_csv(OUTPUT / "fit_comparison_results_scores.csv")
columns = [
    "coordinate_sd", "q", "negative_log_likelihood", "brier", "accuracy",
    "objective", "optimizer_success_flag", "max_abs_selected_gradient",
]
display(fresh_scores.loc[:, columns])
if MODE == "smoke":
    display(Markdown("**Smoke-test estimates only. These are not manuscript estimates.**"))


## 4. Verify internal consistency and inspect reference agreement

Internal consistency means that saved parameters, reconstructed scores, and output files agree with one another. Full mode additionally compares the new results with the stored reference. Agreement with reference tolerances is a separate result from internal consistency.

Fresh optimization need not produce identical coordinates or floating-point values. Differences should be examined alongside training objectives, starts, diagnostics, and held-out scores. Matching rounded scores is not evidence of global optimality.


In [ ]:
from idealpoint.checks import verify_results

verification = verify_results(OUTPUT, DATA, compare_reference=(MODE == "full"))
display(JSON(verification, expanded=False))
assert verification["passed"], "Fresh-result consistency checks failed; inspect the report."

if MODE == "full":
    reference_scores = pd.read_csv(REFERENCE / "fit_comparison_results_scores.csv")
    comparisons = fresh_scores.merge(
        reference_scores,
        on=["coordinate_sd", "q"],
        suffixes=("_fresh", "_reference"),
        validate="one_to_one",
    )
    comparisons["log_loss_difference"] = (
        comparisons["negative_log_likelihood_fresh"]
        - comparisons["negative_log_likelihood_reference"]
    )
    comparisons["brier_difference"] = (
        comparisons["brier_fresh"] - comparisons["brier_reference"]
    )
    display(comparisons.loc[:, [
        "coordinate_sd", "q", "negative_log_likelihood_fresh",
        "negative_log_likelihood_reference", "log_loss_difference", "brier_difference",
    ]])
else:
    print("Reference-score comparison is skipped for smoke mode.")


## 5. Inspect the figures from this run

These figures are generated from the new fits. They use Matplotlib without LaTeX. In smoke mode, the Senate maps and scores are software-test outputs rather than manuscript evidence. The exact matrix construction does not depend on fitting mode.


In [ ]:
generated_images = sorted((OUTPUT / "figures").glob("*.png"))
if not generated_images:
    raise FileNotFoundError("No PNG figures were generated by the study run.")
for image_path in generated_images:
    display(Markdown(f"### {image_path.stem.replace('_', ' ')} ({MODE} run)"))
    display(Image(filename=str(image_path)))


## Interpreting a full rerun

Compare the numerical report with the saved reference before changing any reported value. The empirical study uses one dataset and one entrywise split, and uncertainty intervals are not supplied. Wider coordinate priors can change the predictive comparison, and unresolved convergence remains a limitation of the reference unsquared-distance fits.

Neither party separation in a map nor a held-out ranking establishes general superiority of a model family. The letter's distinction between a Euclidean metric and a transformed response function remains a separate mathematical claim.
